# 🏦 Credit Scoring con Random Forest

**Curso:** Aprendizaje Estadístico  
**Universidad:** UPAO  
**Dataset:** [German Credit Data (credit-g.arff)](https://raw.githubusercontent.com/Waikato/weka-3.8/master/wekadocs/data/credit-g.arff)  
**Instancias:** 1000 (original) → 1400 (balanceadas con SMOTE)  
**Features:** 20 atributos predictivos  
**Target:** `class` (good / bad)  
**Modelo:** Random Forest Classifier  

---

## 📌 Objetivo

Replicar en Python (scikit-learn + imbalanced-learn) el pipeline de Credit Scoring previamente entrenado en **WEKA**, validando que las métricas obtenidas coincidan con las del modelo de referencia:

| Métrica | Valor WEKA (referencia) |
|---|---|
| Accuracy | 81.19% |
| AUC-ROC | 0.895 |
| Coeficiente Gini | 0.790 |
| TP / FP / FN / TN | 173 / 32 / 47 / 168 |


## 📋 Notas Metodológicas

Este notebook replica el pipeline exacto de WEKA. Las siguientes decisiones son **conscientes** y se documentan para claridad académica:

1. **SMOTE aplicado antes del train/test split** — Replicamos el flujo de WEKA. Metodológicamente, lo más correcto sería aplicar SMOTE **solo en training** para evitar *data leakage*. Se documenta como decisión consciente para comparabilidad con WEKA.

2. **Doble estrategia de balanceo** — Se aplica SMOTE (que iguala las clases a 50/50) y adicionalmente `class_weight='balanced'` en el Random Forest. Esto refuerza la atención a la clase minoritaria `bad`, replicando WEKA.

3. **Test set no representativo del mundo real** — Tras SMOTE, el test set queda 50/50. En producción los datos serían ~70/30 (good/bad). Las métricas en test serán **optimistas** vs. producción.

4. **Validación contra WEKA** — Scikit-learn y WEKA implementan Random Forest y SMOTE con diferencias internas. Los resultados pueden variar ±2-3%. El objetivo es **estar en el mismo rango**, no clavar el decimal exacto.

5. **MinMaxScaler en Random Forest** — Matemáticamente innecesario para RF, pero se aplica para **consistencia con el pipeline de WEKA** y reproducibilidad.

6. **OrdinalEncoder en categóricas** — En lugar de OneHotEncoder, para mantener paridad con WEKA (que trata categóricas como ordinales) y simplicidad del pipeline de exportación.

In [ ]:
!pip install -q scikit-learn==1.3.2 imbalanced-learn==0.12.0 pandas==2.1.4 numpy==1.24.3 matplotlib==3.8.2 seaborn==0.13.0 scipy==1.11.4 joblib==1.3.2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import io
import time
import urllib.request

from scipy.io import arff
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from imblearn.over_sampling import SMOTE
from google.colab import files

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ Librerías importadas correctamente')

In [ ]:
# Descargar el ARFF desde GitHub
url = "https://raw.githubusercontent.com/Waikato/weka-3.8/master/wekadocs/data/credit-g.arff"
contenido = urllib.request.urlopen(url).read().decode('utf-8')

# Parsear con scipy
datos, meta = arff.loadarff(io.StringIO(contenido))
df = pd.DataFrame(datos)

# Decodificar columnas tipo bytes a string
for col in df.select_dtypes([object]):
    df[col] = df[col].str.decode('utf-8')

print(f'Shape del dataset: {df.shape}')
print(f'\nTipos de datos:\n{df.dtypes.value_counts()}')
print(f'\nPrimeras 5 filas:')
df.head()

In [ ]:
# Distribución de la variable objetivo
print('═' * 50)
print('DISTRIBUCIÓN DE LA VARIABLE OBJETIVO')
print('═' * 50)
conteo = df['class'].value_counts()
porcentajes = df['class'].value_counts(normalize=True) * 100
for clase in conteo.index:
    print(f'  {clase:>5}: {conteo[clase]:>4} instancias ({porcentajes[clase]:.1f}%)')

# Visualización
fig, ax = plt.subplots(figsize=(7, 5))
colores = ['#28a745' if c == 'good' else '#dc3545' for c in conteo.index]
sns.countplot(x='class', data=df, order=conteo.index, palette=colores, ax=ax)
ax.set_title('Distribución de la variable objetivo (class)', fontsize=14, fontweight='bold')
ax.set_xlabel('Clase')
ax.set_ylabel('Cantidad')
for p in ax.patches:
    altura = int(p.get_height())
    pct = altura / len(df) * 100
    ax.annotate(f'{altura}\n({pct:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 El dataset está desbalanceado: 70% good vs 30% bad')
print('   Por esto aplicaremos SMOTE para balancear las clases.')

In [ ]:
# Separar features (X) y target (y)
X = df.drop(columns=['class'])
y = df['class']

# Detectar automáticamente columnas numéricas y categóricas
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Variables NUMÉRICAS ({len(numeric_cols)}):')
for c in numeric_cols:
    print(f'   • {c}')
print(f'\nVariables CATEGÓRICAS ({len(categorical_cols)}):')
for c in categorical_cols:
    print(f'   • {c}')

# Construir el ColumnTransformer (MinMaxScaler + OrdinalEncoder)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numeric_cols),
        ('cat', OrdinalEncoder(), categorical_cols)
    ],
    remainder='drop'
)

# Paso 1: fit_transform sobre X completo
X_transformed = preprocessor.fit_transform(X)
print(f'\nShape después de preprocessor: {X_transformed.shape}')

# Paso 2: aplicar SMOTE sobre X transformado
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_transformed, y)

print(f'\nShape después de SMOTE: {X_resampled.shape}')
print(f'Distribución después de SMOTE:')
print(pd.Series(y_resampled).value_counts())

# Paso 3: train_test_split 70/30 estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.30, random_state=42, stratify=y_resampled
)

print(f'\n═══════════════════════════════════════')
print(f'Train set: {X_train.shape[0]} instancias')
print(f'Test set:  {X_test.shape[0]} instancias')
print(f'═══════════════════════════════════════')

In [ ]:
# Random Forest con hiperparámetros equivalentes a WEKA
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print('Entrenando Random Forest...')
inicio = time.time()
model.fit(X_train, y_train)
tiempo = time.time() - inicio
print(f'✅ Modelo entrenado en {tiempo:.2f} segundos')

print(f'\nHiperparámetros del modelo:')
print(f'  • n_estimators   = {model.n_estimators}')
print(f'  • max_depth      = {model.max_depth}')
print(f'  • min_samples_leaf = {model.min_samples_leaf}')
print(f'  • class_weight   = {model.class_weight}')
print(f'  • random_state   = {model.random_state}')

In [ ]:
# Predicciones
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # probabilidad de 'bad'

# Calcular métricas
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
gini = 2 * auc - 1

print('═' * 60)
print('   📊 MÉTRICAS EN TEST SET (Python / scikit-learn)')
print('═' * 60)
print(f'   Accuracy:  {accuracy * 100:.2f}%')
print(f'   AUC-ROC:   {auc:.4f}')
print(f'   Gini:      {gini:.4f}')
print('═' * 60)

# Classification report
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['good', 'bad']))

# === COMPARACION CON WEKA ===
print('═' * 60)
print('   🔄 VALIDACIÓN: PYTHON vs WEKA')
print('═' * 60)
print(f'  {"Métrica":<12} {"WEKA":<10} {"Python":<10} {"Δ":<10}')
print(f'  {"-"*42}')
print(f'  {"Accuracy":<12} {"81.19%":<10} {accuracy*100:.2f}%{"":<4} {(accuracy*100-81.19):+.2f}%')
print(f'  {"AUC-ROC":<12} {"0.8950":<10} {auc:.4f}{"":<3} {(auc-0.895):+.4f}')
print(f'  {"Gini":<12} {"0.7900":<10} {gini:.4f}{"":<3} {(gini-0.790):+.4f}')
print('═' * 60)

# Matriz de confusión (heatmap)
cm = confusion_matrix(y_test, y_pred, labels=['good', 'bad'])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['good', 'bad'],
            yticklabels=['good', 'bad'],
            ax=ax, cbar=False,
            annot_kws={'size': 18, 'weight': 'bold'})
ax.set_title('Matriz de Confusión', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Predicho', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
plt.tight_layout()
plt.show()

print('\nMatriz de confusión:')
print(f'  TP (good→good): {cm[0,0]}')
print(f'  FP (bad→good):  {cm[0,1]}')
print(f'  FN (good→bad):  {cm[1,0]}')
print(f'  TN (bad→bad):   {cm[1,1]}')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba, pos_label='bad')
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#1f4e79', lw=2.5, label=f'ROC (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio')
ax.fill_between(fpr, tpr, alpha=0.2, color='#1f4e79')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Curva ROC', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables (Random Forest mide reducción de impureza Gini)
importances = model.feature_importances_
all_features = numeric_cols + categorical_cols  # orden del preprocessor

feat_imp = pd.DataFrame({
    'feature': all_features,
    'importance': importances
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
colores = ['#1f4e79' if i >= len(feat_imp) - 3 else '#4a90b8' for i in range(len(feat_imp))]
ax.barh(feat_imp['feature'], feat_imp['importance'], color=colores)
ax.set_xlabel('Importancia (reducción de impureza)', fontsize=12)
ax.set_title('Importancia de Variables - Random Forest', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('\n🏆 Top 5 features más importantes:')
top5 = feat_imp.sort_values('importance', ascending=False).head()
for i, (_, row) in enumerate(top5.iterrows(), 1):
    print(f'   {i}. {row["feature"]:<22} (importancia: {row["importance"]:.4f})')

print('\n💡 En WEKA, el ranking esperado era: duration, checking_status, age')

In [ ]:
# ============================================================
# EXPORTACIÓN DE ARCHIVOS PARA STREAMLIT
# ============================================================

# 1. Preprocessor (ColumnTransformer fiteado)
joblib.dump(preprocessor, 'preprocessor.pkl')
print('✅ preprocessor.pkl exportado')

# 2. Modelo Random Forest entrenado
joblib.dump(model, 'random_forest_model.pkl')
print('✅ random_forest_model.pkl exportado')

# 3. Nombres de features en el orden correcto
feature_names = numeric_cols + categorical_cols
joblib.dump(feature_names, 'feature_names.pkl')
print('✅ feature_names.pkl exportado')

# 4. Opciones categóricas (para poblar selectboxes en Streamlit)
categorical_options = {col: sorted(df[col].unique().tolist()) for col in categorical_cols}
joblib.dump(categorical_options, 'categorical_options.pkl')
print('✅ categorical_options.pkl exportado')

print('\n════════════════════════════════════════════════════════════')
print('   📦 4 archivos .pkl exportados correctamente')
print('   Descargando a tu equipo...')
print('════════════════════════════════════════════════════════════')

# Descargar archivos
for archivo in ['preprocessor.pkl', 'random_forest_model.pkl',
                'feature_names.pkl', 'categorical_options.pkl']:
    files.download(archivo)

print('\n🎉 ¡Listo! Coloca los 4 archivos .pkl en la misma carpeta que app.py')